# Notebook 03 — Custom-50 Multiclass Evaluation
**D7047E Advanced Deep Learning | Group 14**

Evaluates all multiclass checkpoints on the **custom-50** real-photo dataset (8 styles).

WandB: `adl-crossout-v5 / custom_mc / {model_name}`

## 1. Colab / Drive Setup

In [1]:
import os
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MOUNTED:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
    except Exception as e:
        print(f'Drive mount skipped ({e}).')
print(f'IN_COLAB={IN_COLAB}  DRIVE_MOUNTED={DRIVE_MOUNTED}')

IN_COLAB=False  DRIVE_MOUNTED=False


## 2. Configuration

In [15]:
import sys, os
sys.path.insert(0, '..')

IN_COLAB      = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')

CUSTOM_DIR = '../dataset/custom_50_v5'

if DRIVE_MOUNTED:
    CHECKPOINT_DIR = '/content/drive/MyDrive/adl_checkpoints'
elif IN_COLAB:
    CHECKPOINT_DIR = '/content/adl_checkpoints'
else:
    CHECKPOINT_DIR = '../checkpoints'

IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_WORKERS = 0
WANDB_GROUP = 'custom_mc_v5'

from common import CATEGORIES
NC = len(CATEGORIES)
print(f'Config loaded. NC={NC}  CHECKPOINT_DIR={CHECKPOINT_DIR}')

Config loaded. NC=8  CHECKPOINT_DIR=../checkpoints


## 3. Setup

In [5]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
os.environ['WANDB_API_KEY'] = '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'

In [17]:
import os, gc
import torch
from torch.utils.data import DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
import wandb
from dotenv import load_dotenv

from common import (get_transforms, WANDB_PROJECT, CrossOutDataset,
                    CATEGORIES, rebuild_model)

load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/nagarajan.ganesan/.netrc


Device: cpu


## 4. Load Multiclass Checkpoints

In [19]:
_, val_t = get_transforms(IMG_SIZE)

mc_files = [f for f in os.listdir(CHECKPOINT_DIR)
            if f.startswith('best_mc_') and f.endswith('.pth')]
print(f'Found {len(mc_files)} multiclass checkpoints: {mc_files}')

if not mc_files:
    raise FileNotFoundError('No multiclass checkpoints found in ' + CHECKPOINT_DIR)

# Just read model names from checkpoints — no IAM inference
model_names = []
for fname in sorted(mc_files):
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, fname), map_location='cpu', weights_only=False)
    model_names.append(ckpt['model_name'])
    print(f'  Loaded checkpoint: {fname}  model={ckpt["model_name"]}')

print(f'\nModels to evaluate on Custom-50: {model_names}')

Found 1 multiclass checkpoints: ['best_mc_SimpleCNN.pth']
  Loaded checkpoint: best_mc_SimpleCNN.pth  model=SimpleCNN

Models to evaluate on Custom-50: ['SimpleCNN']


## 5. Custom-50 Multiclass Evaluation

In [21]:
custom_ds     = CrossOutDataset(CUSTOM_DIR, CATEGORIES, val_t)
custom_loader = DataLoader(custom_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Custom-50 multiclass: {len(custom_ds)} images\n')

mc_c50_results = {}
for model_name in model_names:
    safe = model_name.replace('/', '_').replace('-', '_')
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'best_mc_{safe}.pth')
    model = rebuild_model(model_name, NC)
    model.load_state_dict(
        torch.load(ckpt_path, map_location=device, weights_only=False)['model_state_dict']
    )
    model = model.to(device).eval()

    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in custom_loader:
            preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    c50_f1  = f1_score(labels, preds, average='macro', zero_division=0)
    c50_acc = accuracy_score(labels, preds)
    mc_c50_results[model_name] = {'f1': c50_f1, 'acc': c50_acc, 'preds': preds, 'labels': labels}
    print(f'  {model_name:<18} Acc={c50_acc:.4f}  Macro-F1={c50_f1:.4f}')
    del model; torch.cuda.empty_cache(); gc.collect()

best_c50_name = max(mc_c50_results, key=lambda k: mc_c50_results[k]['f1'])
print(f'\nBest on Custom-50: {best_c50_name}')
print()
print(classification_report(
    mc_c50_results[best_c50_name]['labels'],
    mc_c50_results[best_c50_name]['preds'],
    target_names=CATEGORIES, zero_division=0
))

Custom-50 multiclass: 56 images

  SimpleCNN          Acc=0.5536  Macro-F1=0.5343

Best on Custom-50: SimpleCNN

              precision    recall  f1-score   support

       CLEAN       0.39      1.00      0.56         7
 SINGLE_LINE       1.00      0.86      0.92         7
 DOUBLE_LINE       0.83      0.71      0.77         7
    DIAGONAL       0.20      0.29      0.24         7
       CROSS       0.00      0.00      0.00         7
        WAVE       1.00      0.43      0.60         7
     ZIG_ZAG       0.57      0.57      0.57         7
     SCRATCH       0.67      0.57      0.62         7

    accuracy                           0.55        56
   macro avg       0.58      0.55      0.53        56
weighted avg       0.58      0.55      0.53        56



## 6. WandB Logging

In [23]:
def _cm_figure(labels, preds, class_names, title):
    present = sorted(set(labels))
    names   = [class_names[i] for i in present]
    cm = confusion_matrix(labels, preds, labels=present)
    fig, ax = plt.subplots(figsize=(max(5, len(names) * 1.2), max(4, len(names))))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
    ax.set_title(title)
    plt.tight_layout()
    return fig

def _per_class_metrics(labels, preds, class_names):
    metrics = {}
    for i in sorted(set(labels)):
        cat   = class_names[i]
        lbl_i = [1 if l == i else 0 for l in labels]
        prd_i = [1 if p == i else 0 for p in preds]
        metrics[f'{cat}_precision'] = precision_score(lbl_i, prd_i, zero_division=0)
        metrics[f'{cat}_recall']    = recall_score(lbl_i, prd_i, zero_division=0)
        metrics[f'{cat}_f1']        = f1_score(lbl_i, prd_i, zero_division=0)
    return metrics

print('Logging to WandB...')
for name in model_names:
    r = mc_c50_results[name]
    run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f'multiclass_{name}',
                     config=dict(model=name, task='custom50_multiclass'), reinit=True)
    log_dict = {
        'custom50_acc':       r['acc'],
        'custom50_macro_f1':  r['f1'],
        'custom50_precision': precision_score(r['labels'], r['preds'], average='macro', zero_division=0),
        'custom50_recall':    recall_score(r['labels'], r['preds'], average='macro', zero_division=0),
        'confusion_matrix':   wandb.Image(_cm_figure(
            r['labels'], r['preds'], CATEGORIES,
            f'Custom-50 Multiclass CM — {name}')),
    }
    log_dict.update(_per_class_metrics(r['labels'], r['preds'], CATEGORIES))
    wandb.log(log_dict)
    plt.close('all')
    run.finish()
print('All results logged to WandB.')

Logging to WandB...


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


CLEAN_f1,▁
CLEAN_precision,▁
CLEAN_recall,▁
CROSS_f1,▁
CROSS_precision,▁
CROSS_recall,▁
DIAGONAL_f1,▁
DIAGONAL_precision,▁
DIAGONAL_recall,▁
DOUBLE_LINE_f1,▁
+18,...


All results logged to WandB.


## 8. Confusion Matrices — All Multiclass Models

In [25]:
n_models = len(mc_c50_results)
cols = min(n_models, 3)
rows = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
axes = np.array(axes).flatten() if n_models > 1 else [axes]

for i, model_name in enumerate(sorted(mc_c50_results.keys())):
    res     = mc_c50_results[model_name]
    present = sorted(set(res['labels']))
    names   = [CATEGORIES[j] for j in present]
    cm  = confusion_matrix(res['labels'], res['preds'], labels=present)
    f1  = f1_score(res['labels'], res['preds'], average='macro', zero_division=0)
    ConfusionMatrixDisplay(cm, display_labels=names).plot(
        ax=axes[i], xticks_rotation=45, colorbar=False, cmap='Blues')
    marker = ' ★' if model_name == best_c50_name else ''
    axes[i].set_title(f'{model_name}{marker}\nMacro-F1={f1:.3f}', fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Custom-50 Confusion Matrices — All Multiclass Models', fontsize=12)
plt.tight_layout()
plt.savefig('custom50_mc_all_cm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: custom50_mc_all_cm.png')

Saved: custom50_mc_all_cm.png


/var/folders/34/gp_zqq8s68l08cbyqd89g9x80000gn/T/ipykernel_61092/1553053805.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Sample Predictions Grid — Best Model

In [27]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])
N_SAMPLES = 3

safe = best_c50_name.replace('/', '_').replace('-', '_')
best_model = rebuild_model(best_c50_name, NC)
best_model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, f'best_mc_{safe}.pth'),
               map_location=device, weights_only=False)['model_state_dict']
)
best_model = best_model.to(device).eval()

from PIL import Image
fig, axes = plt.subplots(len(CATEGORIES), N_SAMPLES,
                          figsize=(N_SAMPLES * 3, len(CATEGORIES) * 2))

for row, cat in enumerate(CATEGORIES):
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        for col in range(N_SAMPLES): axes[row][col].axis('off')
        continue
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith('.png')])[:N_SAMPLES]
    for col, fname in enumerate(files):
        raw    = Image.open(os.path.join(folder, fname)).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            probs    = torch.softmax(best_model(tensor), dim=1)[0]
            pred_idx = probs.argmax().item()
        pred_cat = CATEGORIES[pred_idx]
        conf     = probs[pred_idx].item()
        disp = (val_t(raw) * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]).clamp(0, 1)
        axes[row][col].imshow(disp.permute(1, 2, 0).numpy())
        axes[row][col].axis('off')
        color = 'green' if pred_cat == cat else 'red'
        axes[row][col].set_title(f'{pred_cat}\n{conf:.0%}', fontsize=7, color=color)
    for col in range(len(files), N_SAMPLES):
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'GT: {cat}', fontsize=8, rotation=0, labelpad=65, va='center')

fig.suptitle(f'Custom-50 predictions — {best_c50_name} (green=correct, red=wrong)', fontsize=11)
plt.tight_layout()
plt.savefig('custom50_mc_predictions.png', dpi=150)
plt.show()
print('Saved: custom50_mc_predictions.png')
del best_model; torch.cuda.empty_cache(); gc.collect()

Saved: custom50_mc_predictions.png


/var/folders/34/gp_zqq8s68l08cbyqd89g9x80000gn/T/ipykernel_61092/329567119.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


8068

## 10. Misclassified Examples — Best Model

In [29]:
from PIL import Image

safe = best_c50_name.replace('/', '_').replace('-', '_')
best_model = rebuild_model(best_c50_name, NC)
best_model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, f'best_mc_{safe}.pth'),
               map_location=device, weights_only=False)['model_state_dict']
)
best_model = best_model.to(device).eval()

wrong = []
for cat_idx, cat in enumerate(CATEGORIES):
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        continue
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path = os.path.join(folder, fname)
        raw    = Image.open(img_path).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            probs    = torch.softmax(best_model(tensor), dim=1)[0]
            pred_idx = probs.argmax().item()
        if pred_idx != cat_idx:
            wrong.append((img_path, cat, CATEGORIES[pred_idx], probs[pred_idx].item(), raw))

print(f'{best_c50_name} — {len(wrong)} misclassified / {len(custom_ds)} Custom-50 images\n')
for img_path, true_cls, pred_cls, conf, _ in wrong:
    print(f'  True: {true_cls:<14}  Pred: {pred_cls:<14}  Conf: {conf:.2f}  '
          f'File: {os.path.basename(img_path)}')

if wrong:
    cols = min(5, len(wrong))
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3.4))
    axes = np.array(axes).flatten()

    for ax, (img_path, true_cls, pred_cls, conf, raw) in zip(axes, wrong):
        disp = (val_t(raw) * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]).clamp(0, 1)
        ax.imshow(disp.permute(1, 2, 0).numpy())
        ax.set_title(f'True:  {true_cls}\nPred:  {pred_cls}\nConf: {conf:.0%}',
                     fontsize=7, color='red')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor('red'); spine.set_linewidth(2)

    for ax in axes[len(wrong):]:
        ax.set_visible(False)

    plt.suptitle(f'Misclassified — {best_c50_name} on Custom-50 ({len(wrong)} errors)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('custom50_mc_misclassified.png', dpi=150)
    plt.show()
    print('Saved: custom50_mc_misclassified.png')
else:
    print('No misclassified images — perfect score on Custom-50!')

del best_model; torch.cuda.empty_cache(); gc.collect()

SimpleCNN — 25 misclassified / 56 Custom-50 images

  True: SINGLE_LINE     Pred: CLEAN           Conf: 0.56  File: single_line_03.png
  True: DOUBLE_LINE     Pred: SCRATCH         Conf: 0.99  File: double_line_03.png
  True: DOUBLE_LINE     Pred: SCRATCH         Conf: 0.44  File: double_line_04.png
  True: DIAGONAL        Pred: ZIG_ZAG         Conf: 0.44  File: diagonal_00.png
  True: DIAGONAL        Pred: ZIG_ZAG         Conf: 0.86  File: diagonal_03.png
  True: DIAGONAL        Pred: CLEAN           Conf: 0.43  File: diagonal_04.png
  True: DIAGONAL        Pred: ZIG_ZAG         Conf: 1.00  File: diagonal_05.png
  True: DIAGONAL        Pred: CLEAN           Conf: 0.51  File: diagonal_06.png
  True: CROSS           Pred: DIAGONAL        Conf: 1.00  File: cross_00.png
  True: CROSS           Pred: DIAGONAL        Conf: 1.00  File: cross_01.png
  True: CROSS           Pred: DIAGONAL        Conf: 1.00  File: cross_02.png
  True: CROSS           Pred: DIAGONAL        Conf: 0.91  File: cros

/var/folders/34/gp_zqq8s68l08cbyqd89g9x80000gn/T/ipykernel_61092/2216334425.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


85